# Test Full CV Pipeline on Kaggle

Notebook nay chay lien mach Module 1 segmentation, Module 2 attribute recognition, Module 3 OCR va Module 4 fusion tren mot anh.

Truoc khi chay, attach bon Kaggle Dataset: checkpoint segmentation, artifact Attribute last-block, anh test, va SQLite database da seed. Notebook clone source code tu GitHub vao `/kaggle/working`. PaddleOCR chay trong process GPU rieng de khong xung dot voi PyTorch.

In [ ]:
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
# Branch nay phai da duoc push va chua ca CV, RAG/DDI va end-to-end orchestration.
GIT_BRANCH = 'dev'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

# Sua cac path nay theo ten Kaggle Dataset cua ban.
SEGMENTATION_WEIGHTS = Path(
    '/kaggle/input/datasets/nnphuchcmus/pill-segmentation-model/yolov11m_seg_mediseg_full_finetune_v1.pt'
)
IMAGE_PATH = Path('/kaggle/input/datasets/nnphuchcmus/same-bg/Screenshot 2026-08-13 011709.png')

# Thu muc nay phai chua: best.pt, label_mapping.json, optimal_thresholds.json, model_config.yaml.
ATTRIBUTE_ARTIFACT_DIR = Path(
    '/kaggle/input/datasets/nnphuchcmus/attrubute-artifact/kaggle_uploads/attribute_resnet18_last_blocks_finetune'
)

# Tat ca artifact inference se ghi vao working directory co quyen ghi.
OUTPUT_DIR = Path('/kaggle/working/pill_cv_outputs')

REQUEST_ID = 'req_kaggle_001'
SESSION_ID = 'kaggle_full_cv_test'
IMAGE_ID = IMAGE_PATH.stem

print('REPO_DIR:', REPO_DIR)
print('SEGMENTATION_WEIGHTS:', SEGMENTATION_WEIGHTS)
print('ATTRIBUTE_ARTIFACT_DIR:', ATTRIBUTE_ARTIFACT_DIR)
print('IMAGE_PATH:', IMAGE_PATH)

In [ ]:
# Kaggle Notebook Settings phai bat Internet de clone GitHub.
import subprocess

if not REPO_DIR.is_dir():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, GIT_REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Repository da ton tai, bo qua clone: {REPO_DIR}')
    print(f'Xoa REPO_DIR va chay lai cell nay neu can lay commit moi cua {GIT_BRANCH}.')

print('Repository ready:', REPO_DIR)


In [ ]:
# Kernel chinh chay segmentation bang PyTorch/Ultralytics GPU.
# OCR duoc cai theo dung bo package cua PaddleOCR_baseline sau khi segmentation xong.
# --no-deps giu nguyen torch/torchvision CUDA ma Kaggle da cung cap.
%pip install -q --no-deps ultralytics==8.3.253 pyyaml==6.0.2

import sys
import torch

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Torch CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch khong thay GPU. Hay bat GPU accelerator tren Kaggle.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Kiem tra tai nguyen truoc khi import code project.
if not REPO_DIR.is_dir():
    raise FileNotFoundError(
        f'Repo khong ton tai: {REPO_DIR}. Chay lai cell clone va kiem tra Internet setting.'
    )
if not SEGMENTATION_WEIGHTS.is_file():
    raise FileNotFoundError(f'Khong tim thay checkpoint: {SEGMENTATION_WEIGHTS}')
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
for artifact_name in ('best.pt', 'label_mapping.json', 'optimal_thresholds.json', 'model_config.yaml'):
    artifact_path = ATTRIBUTE_ARTIFACT_DIR / artifact_name
    if not artifact_path.is_file():
        raise FileNotFoundError(f'Khong tim thay Attribute artifact: {artifact_path}')

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pill_safety.cv.segmentation import SegmentationConfig, SegmentationPredictor
from pill_safety.schemas import SegmentationInferenceRequest

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project source:', SRC_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

with Image.open(IMAGE_PATH) as source:
    image = source.convert('RGB')

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title(f'Input image: {IMAGE_PATH.name}')
plt.axis('off')
plt.show()


In [ ]:
# Tao request truc tiep tu IMAGE_PATH; khong can tao request.json.
request = SegmentationInferenceRequest(
    request_id=REQUEST_ID,
    session_id=SESSION_ID,
    image_id=IMAGE_ID,
    image_path=str(IMAGE_PATH),
)

config = SegmentationConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'
).with_weights_path(SEGMENTATION_WEIGHTS).with_output_dir(OUTPUT_DIR)

# Config YAML da dung device=auto: Ultralytics se dung GPU neu Kaggle cap GPU.
predictor = SegmentationPredictor(config=config)
artifacts = predictor.predict_with_artifacts(request)
result = artifacts.output.model_dump(mode='json')

print('Detected instances:', len(result['instances']))
print('Schema JSON:', artifacts.schema_json_path)
print('Overlay:', artifacts.overlay_path)


In [ ]:
# Hien thi overlay va thong tin tong quat.
if artifacts.overlay_path is not None and artifacts.overlay_path.exists():
    with Image.open(artifacts.overlay_path) as source:
        overlay = source.convert('RGB')
    plt.figure(figsize=(12, 8))
    plt.imshow(overlay)
    plt.title('Segmentation overlay')
    plt.axis('off')
    plt.show()

display({
    'image_quality': result['image_quality'],
    'instance_count': len(result['instances']),
})


In [ ]:
# Hien thi clean mask, color crop va shape crop cua tung vien.
instances = result['instances']
if not instances:
    print('Khong phat hien vien nao. Kiem tra checkpoint, confidence_threshold va anh input.')
else:
    figure, axes = plt.subplots(len(instances), 3, figsize=(15, 5 * len(instances)))
    if len(instances) == 1:
        axes = [axes]

    for row, instance in zip(axes, instances):
        with Image.open(instance['mask_path']) as source:
            mask = source.convert('L')
        with Image.open(instance['color_crop_path']) as source:
            color_crop = source.convert('RGB')
        with Image.open(instance['shape_crop_path']) as source:
            shape_crop = source.convert('RGB')

        row[0].imshow(mask, cmap='gray')
        row[0].set_title(f"{instance['instance_id']} mask")
        row[0].axis('off')
        row[1].imshow(color_crop)
        row[1].set_title(
            f"{instance['instance_id']} color crop | conf={instance['segmentation']['confidence']:.3f}"
        )
        row[1].axis('off')
        row[2].imshow(shape_crop)
        row[2].set_title(f"{instance['instance_id']} shape crop")
        row[2].axis('off')

    plt.tight_layout()
    plt.show()

    for instance in instances:
        print('\n', instance['instance_id'])
        display({
            'bbox_xyxy': instance['bbox_xyxy'],
            'segmentation': instance['segmentation'],
            'quality_flags': instance['quality_flags'],
            'mask_path': instance['mask_path'],
            'color_crop_path': instance['color_crop_path'],
            'shape_crop_path': instance['shape_crop_path'],
            'ocr_crop_path': instance['ocr_crop_path'],
            'crop_path': instance['crop_path'],
        })


In [ ]:
# Module 1 JSON nay la input de tao request cho Attribute va OCR o buoc sau.
import json

print(json.dumps(result, indent=2, ensure_ascii=False))
print('\nSaved JSON:', artifacts.schema_json_path)


## Test Module 2 Attribute Recognition from Segmentation Crops

Module 2 nhan `color_crop_path` cho color head va `shape_crop_path` cho shape head. ResNet18 duoc chay tren RGB crop theo dung transform validation da train; mask chi la contract chung voi cac module, khong bi dung de mask anh va gay distribution shift.

In [ ]:
# Khoi tao Module 2 mot lan va tai su dung cho moi pill crop.
from dataclasses import replace

from pill_safety.cv.attribute.config import AttributeInferenceConfig
from pill_safety.cv.attribute.predictors import AttributePredictor
from pill_safety.schemas import AttributeInferenceRequest

attribute_config = AttributeInferenceConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'attribute.yaml'
).resolve_paths(REPO_DIR)
attribute_config = replace(
    attribute_config,
    weights_path=ATTRIBUTE_ARTIFACT_DIR / 'best.pt',
    label_mapping_path=ATTRIBUTE_ARTIFACT_DIR / 'label_mapping.json',
    color_thresholds_path=ATTRIBUTE_ARTIFACT_DIR / 'optimal_thresholds.json',
    model_config_path=ATTRIBUTE_ARTIFACT_DIR / 'model_config.yaml',
    output_dir=OUTPUT_DIR,
)
attribute_predictor = AttributePredictor(config=attribute_config)

# In raw softmax cua shape de kiem tra model dang chon class index nao.
# Cac bien shape_probabilities/shape_names chi ton tai trong ham debug nay.
import torch

def debug_shape_prediction(predictor, crop_path):
    with Image.open(crop_path) as source:
        image = source.convert('RGB')
    tensor = predictor.transform(image).unsqueeze(0).to(predictor.device)
    with torch.inference_mode():
        logits = predictor.model(tensor, task_type='shape')
        probabilities = torch.softmax(logits, dim=1)[0].detach().cpu()
    shape_names = predictor.label_mapping['shape']
    ranked_probabilities, ranked_indices = torch.sort(
        probabilities, descending=True
    )
    predicted_index = int(ranked_indices[0].item())
    print('RAW SHAPE DEBUG')
    print('probabilities:', {
        shape_names[index]: round(float(probability), 4)
        for index, probability in enumerate(probabilities)
    })
    print('predicted_index:', predicted_index)
    print('predicted_label:', shape_names[predicted_index])
    print('top3:', [
        (shape_names[int(index.item())], round(float(probability.item()), 4))
        for probability, index in zip(
            ranked_probabilities[:3], ranked_indices[:3]
        )
    ])

attribute_outputs = []
attribute_artifacts_by_instance = {}

for instance in artifacts.output.instances:
    attribute_request = AttributeInferenceRequest(
        request_id=artifacts.output.request_id,
        session_id=artifacts.output.session_id,
        image_id=artifacts.output.image_id,
        instance_id=instance.instance_id,
        instance_token=instance.instance_token,
        crop_path=instance.color_crop_path,
        color_crop_path=instance.color_crop_path,
        shape_crop_path=instance.shape_crop_path,
        mask_path=instance.mask_path,
    )
    attribute_artifacts = attribute_predictor.predict_with_artifacts(attribute_request)
    debug_shape_prediction(attribute_predictor, instance.shape_crop_path)
    attribute_outputs.append(attribute_artifacts.output)
    attribute_artifacts_by_instance[instance.instance_id] = attribute_artifacts

print('Attribute completed:', len(attribute_outputs), 'pill(s)')
for attribute_output in attribute_outputs:
    display({
        'instance_id': attribute_output.instance_id,
        'shape': attribute_output.shape.model_dump(mode='json'),
        'color': attribute_output.color.model_dump(mode='json'),
        'scoreline_placeholder': attribute_output.scoreline.model_dump(mode='json'),
        'schema_json_path': str(
            attribute_artifacts_by_instance[attribute_output.instance_id].schema_json_path
        ),
    })


## Test Module 3 OCR from Segmentation Crops

Cac cell ben duoi dung truc tiep `ocr_crop_path` va clean `mask_path` do Module 1 vua sinh ra. OCR khong can upload lai anh va khong can tao file request JSON.

In [ ]:
# Tao PaddleOCR GPU environment rieng de khong thay NCCL/CUDA cua PyTorch.
# Cell nay chi cai mot lan; cac lan chay sau se tai su dung OCR_VENV_DIR.
import gc
import subprocess

# Module 1 va Module 2 da hoan tat; giai phong hai model PyTorch de PaddleOCR co du VRAM.
if 'predictor' in globals():
    del predictor
if 'attribute_predictor' in globals():
    del attribute_predictor
gc.collect()
torch.cuda.empty_cache()
print('Released YOLO model before starting PaddleOCR GPU process.')

OCR_VENV_DIR = Path('/kaggle/working/paddleocr_gpu_venv')
OCR_PYTHON = OCR_VENV_DIR / 'bin' / 'python'
OCR_READY_FILE = OCR_VENV_DIR / '.ocr_environment_ready'

def run_checked(command, label):
    print(f'\n>>> {label}')
    print(' '.join(str(part) for part in command))
    # Khong capture stdout/stderr de Kaggle hien tien do pip theo thoi gian thuc.
    completed = subprocess.run(command)
    if completed.returncode != 0:
        raise RuntimeError(f'{label} failed (exit={completed.returncode})')
    return completed

def install_ocr_environment():
    # Kaggle /usr/bin/python3 khong co ensurepip, nen dung virtualenv thay cho stdlib venv.
    # --clear tu sua thu muc bi tao do dang tu lan chay truoc.
    virtualenv_check = subprocess.run(
        [sys.executable, '-m', 'virtualenv', '--version'],
        text=True, capture_output=True,
    )
    if virtualenv_check.returncode != 0:
        run_checked(
            [
                sys.executable, '-m', 'pip', 'install',
                'virtualenv==20.31.2',
            ],
            'Install virtualenv for the OCR subprocess',
        )
    run_checked(
        [sys.executable, '-m', 'virtualenv', '--clear', str(OCR_VENV_DIR)],
        'Create isolated PaddleOCR environment',
    )
    # Paddle imports setuptools during startup; virtualenv may not seed it on Kaggle.
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            '--upgrade', 'pip', 'setuptools', 'wheel',
        ],
        'Bootstrap OCR environment packaging tools',
    )
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            'paddlepaddle-gpu==3.0.0',
            '--index-url', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
        ],
        'Install Paddle GPU',
    )
    run_checked(
        [
            str(OCR_PYTHON), '-m', 'pip', 'install',
            'paddleocr==3.0.3', 'paddlex==3.0.3',
            'langchain==0.3.27', 'langchain-community==0.3.27',
            'langchain-text-splitters==0.3.9',
            'opencv-python-headless==4.10.0.84', 'numpy==1.26.4',
            'pillow==11.0.0', 'pydantic==2.9.2', 'pyyaml==6.0.2',
        ],
        'Install PaddleOCR dependencies',
    )


OCR_CHECK_COMMAND = [
    str(OCR_PYTHON), '-c',
    (
        "from importlib.metadata import version; "
        "import setuptools; import paddle, paddleocr, paddlex; "
        "from langchain.docstore.document import Document; "
        "assert version('paddlepaddle-gpu') == '3.0.0'; "
        "assert version('paddleocr') == '3.0.3'; "
        "assert version('paddlex') == '3.0.3'; "
        "assert version('langchain') == '0.3.27'; "
        "assert paddle.device.is_compiled_with_cuda(); "
        "paddle.set_device('gpu:0'); "
        "print('Paddle', paddle.__version__); "
        "print('PaddleOCR', paddleocr.__version__); "
        "print('PaddleX', version('paddlex')); "
        "print('Paddle device', paddle.get_device())"
    ),
]

def check_ocr_environment():
    if not OCR_PYTHON.is_file():
        return None
    return subprocess.run(
        OCR_CHECK_COMMAND, text=True, capture_output=True
    )


ocr_env_check = check_ocr_environment() if OCR_READY_FILE.is_file() else None
if ocr_env_check is None or ocr_env_check.returncode != 0:
    if OCR_READY_FILE.exists():
        OCR_READY_FILE.unlink()
    print('Building or repairing isolated PaddleOCR GPU environment...')
    install_ocr_environment()
    ocr_env_check = check_ocr_environment()

if ocr_env_check is None:
    raise RuntimeError(f'OCR Python was not created: {OCR_PYTHON}')
print(ocr_env_check.stdout)
if ocr_env_check.returncode != 0:
    raise RuntimeError(
        'PaddleOCR GPU environment check failed after rebuild:\n'
        f'STDOUT:\n{ocr_env_check.stdout}\nSTDERR:\n{ocr_env_check.stderr}'
    )
OCR_READY_FILE.touch()
print('OCR GPU environment: READY')


In [ ]:
# OCR chay trong process Paddle GPU rieng; kernel PyTorch khong import Paddle.
OCR_OUTPUT_DIR = OUTPUT_DIR / 'predictions' / 'ocr'
OCR_REQUEST_DIR = OUTPUT_DIR / 'requests' / 'ocr'
OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_REQUEST_DIR.mkdir(parents=True, exist_ok=True)
OCR_RUNNER = REPO_DIR / 'inference' / 'cv_ocr' / 'run_ocr.py'
OCR_CONFIG_PATH = REPO_DIR / 'configs' / 'inference' / 'ocr.yaml'

if not OCR_RUNNER.is_file():
    raise FileNotFoundError(f'Khong tim thay OCR runner: {OCR_RUNNER}')
print('OCR output directory:', OCR_OUTPUT_DIR)


In [ ]:
# Tao request JSON tu crop/mask Module 1 va chay OCR tren Paddle GPU process.
# Quy tac nay phai trung voi _safe_directory_name() trong OCRPredictor.
import re

def ocr_artifact_directory_name(value):
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('._')
    return safe or 'pill'

ocr_results_by_instance = {}
ocr_artifact_paths_by_instance = {}

if not result['instances']:
    print('Khong co segmentation instance, bo qua OCR.')
else:
    for instance in result['instances']:
        instance_id = instance['instance_id']
        ocr_request_payload = {
            'request_id': result['request_id'],
            'session_id': result['session_id'],
            'image_id': result['image_id'],
            'instance_id': instance_id,
            'instance_token': instance['instance_token'],
            'crop_path': instance['ocr_crop_path'],
            'mask_path': instance['mask_path'],
        }
        request_path = OCR_REQUEST_DIR / f'{instance_id}_request.json'
        request_path.write_text(
            json.dumps(ocr_request_payload, indent=2, ensure_ascii=False),
            encoding='utf-8',
        )

        print(f'Running OCR on Paddle GPU: {instance_id}')
        completed = subprocess.run(
            [
                str(OCR_PYTHON), str(OCR_RUNNER),
                '--request', str(request_path),
                '--config', str(OCR_CONFIG_PATH),
                '--output-dir', str(OCR_OUTPUT_DIR),
            ],
            cwd=str(REPO_DIR), text=True, capture_output=True,
        )
        if completed.returncode != 0:
            raise RuntimeError(
                f'OCR failed for {instance_id}:\nSTDOUT:\n{completed.stdout}\n'
                f'STDERR:\n{completed.stderr}'
            )
        instance_dir = (
            OCR_OUTPUT_DIR
            / ocr_artifact_directory_name(result['request_id'])
            / ocr_artifact_directory_name(result['image_id'])
            / ocr_artifact_directory_name(instance_id)
        )
        artifact_paths = {
            'schema': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_ocr_schema.json',
            'debug': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_final_result.json',
            'overlay': instance_dir / f'{ocr_artifact_directory_name(instance_id)}_final_overlay.jpg',
        }
        if not artifact_paths['schema'].is_file():
            raise FileNotFoundError(
                f'OCR process thanh cong nhung khong tao schema: {artifact_paths["schema"]}'
            )
        ocr_result = json.loads(artifact_paths['schema'].read_text(encoding='utf-8'))
        ocr_results_by_instance[instance_id] = ocr_result
        ocr_artifact_paths_by_instance[instance_id] = artifact_paths

print('OCR completed:', len(ocr_results_by_instance), 'pill(s)')


In [ ]:
# Hien thi crop, final overlay, final answer va scoreline cua tung vien.
for instance in result['instances']:
    instance_id = instance['instance_id']
    ocr_result = ocr_results_by_instance.get(instance_id)
    artifact_paths = ocr_artifact_paths_by_instance.get(instance_id)
    if ocr_result is None or artifact_paths is None:
        continue

    with Image.open(instance['ocr_crop_path']) as source:
        crop = source.convert('RGB')

    figure, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(crop)
    axes[0].set_title(f'{instance_id} crop from Module 1')
    axes[0].axis('off')

    if artifact_paths['overlay'].is_file():
        with Image.open(artifact_paths['overlay']) as source:
            overlay = source.convert('RGB')
        axes[1].imshow(overlay)
        axes[1].set_title('Module 3 final OCR overlay')
    else:
        axes[1].text(0.5, 0.5, 'No OCR overlay: no usable text', ha='center', va='center')
        axes[1].set_title('Module 3 final OCR overlay')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    display({
        'instance_id': instance_id,
        'imprint_visibility': ocr_result['imprint_visibility'],
        'final_text': ocr_result['imprint']['raw'],
        'final_confidence': ocr_result['imprint']['confidence'],
        'scoreline': ocr_result['scoreline'],
        'normalized_candidates': ocr_result['imprint']['normalized_candidates'],
        'schema_json_path': str(artifact_paths['schema']),
        'debug_json_path': str(artifact_paths['debug']),
    })


In [ ]:
# In Module 3 JSON de kiem tra schema va instance_token truoc khi fusion Module 4.
ocr_outputs = []
for instance_id, ocr_result in ocr_results_by_instance.items():
    ocr_outputs.append(ocr_result)
    print(f'\n===== Module 3 OCR JSON: {instance_id} =====')
    print(json.dumps(ocr_result, indent=2, ensure_ascii=False))

print('\nOCR schema artifacts:')
for artifact_paths in ocr_artifact_paths_by_instance.values():
    print(artifact_paths['schema'])


## Module 4: Fuse CV Outputs

Cell nay chi fusion cac JSON da co. No kiem tra `request_id`, `image_id`, `instance_id` va `instance_token`, sau do thay placeholder scoreline cua Module 2 bang ket qua OCR cua Module 3.

In [ ]:
from pill_safety.cv.pipeline import CVPipelineAssembler, CVPipelineConfig
from pill_safety.schemas import CVPipelineInput

pipeline_config = CVPipelineConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'cv_pipeline.yaml'
).with_output_dir(OUTPUT_DIR)
pipeline_artifacts = CVPipelineAssembler(config=pipeline_config).predict_with_artifacts(
    CVPipelineInput(
        segmentation_output=artifacts.output,
        attribute_outputs=attribute_outputs,
        ocr_outputs=ocr_outputs,
    )
)
cv_output = pipeline_artifacts.output.model_dump(mode='json')

print('CV fusion completed:', len(cv_output['pills']), 'pill(s)')
print('Final CV schema:', pipeline_artifacts.schema_json_path)
print(json.dumps(cv_output, indent=2, ensure_ascii=False))


## Module 5-8: Retrieval, DDI, and Grounded LLM Report

Attach a Kaggle dataset containing exactly one seeded `pill_safety.db`. The next cell discovers it automatically, then consumes the `cv_output_v1` generated by Module 4, identifies only accepted candidates, checks DDI, and builds a grounded fallback report.

In [ ]:
# Tu dong tim SQLite database da attach vao Kaggle Input.
# Dataset can nam trong nhieu cap thu muc; file bat buoc co ten pill_safety.db.
DATABASE_FILE_NAME = 'pill_safety.db'
database_candidates = sorted(Path('/kaggle/input').rglob(DATABASE_FILE_NAME))
if len(database_candidates) == 0:
    raise FileNotFoundError(
        f'Khong tim thay {DATABASE_FILE_NAME} trong /kaggle/input. Attach Kaggle dataset chua SQLite database da seed.'
    )
if len(database_candidates) > 1:
    found = '\n'.join(str(path) for path in database_candidates)
    raise RuntimeError(
        f'Tim thay nhieu file {DATABASE_FILE_NAME}; chi attach mot database dataset de tranh dung nham:\n{found}'
    )
DATABASE_PATH = database_candidates[0]
if DATABASE_PATH.stat().st_size == 0:
    raise ValueError(f'Database rong: {DATABASE_PATH}')
DATABASE_URL = f'sqlite:///{DATABASE_PATH.as_posix()}'
MARKET = 'US'
KNOWN_DRUG_NAMES = []
LLM_PROVIDER = 'fallback'

print('SQLite database:', DATABASE_PATH)
print('DATABASE_URL:', DATABASE_URL)
print('LLM_PROVIDER:', LLM_PROVIDER)

import os
import pandas as pd
from pill_safety.core.config import get_settings

# Set environment before importing SessionLocal because its engine is created at import time.
os.environ['DATABASE_URL'] = DATABASE_URL
os.environ['LLM_PROVIDER'] = LLM_PROVIDER
get_settings.cache_clear()

from pill_safety.database.session import SessionLocal
from pill_safety.rag.orchestration import EndToEndPostCvPipeline

END_TO_END_DIR = OUTPUT_DIR / 'reports' / cv_output['request_id']
db = SessionLocal()
try:
    end_to_end_artifacts = EndToEndPostCvPipeline.from_database_session(
        db,
        llm_provider=LLM_PROVIDER,
    ).run_with_artifacts(
        cv_output,
        output_dir=END_TO_END_DIR,
        market=MARKET,
        known_drug_names=KNOWN_DRUG_NAMES,
    )
finally:
    db.close()

end_to_end_result = end_to_end_artifacts.output
print('End-to-end completed:', len(end_to_end_result['pill_summary']), 'pill(s)')
for name, path in end_to_end_artifacts.paths.items():
    print(f'{name}: {path}')

display(pd.DataFrame(end_to_end_result['pill_summary']))
print('\n===== Grounded LLM report =====')
print(end_to_end_result['llm_report']['formatted_report_text'])
